# IndetBench · parte B: p(UNKNOWN) nei modelli reasoning (Colab, A100-80GB)

Versione del notebook: **R-notebook-1** (15 novembre 2026). Esegue il **Protocollo parte B v1**, approvato da Simona il 15 novembre prima di qualsiasi dato. Benchmark e codice di calcolo vengono dalla repository `Sva76/indet-bench` al commit `2aa1f62`, con lo stesso calcolo di W, soglia e report delle parti precedenti.

**Che cosa fa.** Usa le stesse 800 coppie B/C congelate in prosa (set PN). Il modello ragiona prima di rispondere, e p(UNKNOWN) si misura **al primo token della risposta dopo `</think>`**.
- **Ragionamento:** campionamento con i parametri degli autori, seme fisso, budget di 4096 token. Se il ragionamento non si chiude entro il budget, il notebook lo chiude e marca la riga.
- **Risposta:** fino a 16 token dopo `</think>`, in modo greedy.
- **Condizione spenta:** per Qwen3-8B e Qwen3-32B si rimisura nella stessa pipeline (batch, sdpa), con la misura del Test U2. Il confronto acceso/spento (HR2) è così a pesi e impostazioni fissi.

**Controlli e regole fissati prima dei dati**
- Controllo della pipeline: con il ragionamento spento, Qwen3-8B deve riprodurre i 24 valori del Test U2 entro 0.01 con 24/24 risposte uguali. Se fallisce, il completo non parte.
- Regola sulla massa: massa mediana di YES+NO+UNKNOWN < 0.5 al primo token della risposta → NON MISURABILE. Si controlla prima sulle 24 righe di controllo, poi sull'insieme completo.
- Copertura: servono almeno 760 coppie in cui B e C chiudono il ragionamento da sole. Sotto questa soglia il risultato del modello è descrittivo.
- Ripetibilità: le prime 20 storie si ripetono con due semi in più.
- Baseline nuove (descrittive): lunghezza del ragionamento e ragionamento troncato.

**Come leggere i risultati.** Tutti i risultati sono **ESPLORATIVI**. HR2 vale solo per Qwen3, dove i pesi sono gli stessi; Phi-4-reasoning (contro Phi-4 del 14 novembre) e R1-Distill sono descrittivi.

**Come partire**
1. Runtime → **A100**, poi controlla con `!nvidia-smi` che sia la **A100-SXM4-80GB**.
2. Secrets: `HF_TOKEN` con accesso al notebook. `testU2_risultati.zip` deve essere in MyDrive (cella 4b).
3. **Prova:** `MODALITA = "prova"`, `MODELLI = ["qwen3_8b"]`, poi `AVVIA_GPU = True`. La prova usa 10 storie (80 righe), acceso e spento. Misura i tempi e stampa la stima per il completo.
4. **Completo** (solo dopo la verifica della prova): `MODALITA = "completo"` e `MODELLI = ["qwen3_8b", "qwen3_32b", "phi4_reasoning"]`. Aggiungi `"r1_llama_8b"` in coda se vuoi il modello facoltativo. Qwen3-8B va per primo.

Le uscite vanno in `MyDrive/indetbench_reasoning/<modalità>/`, una sola copia. I ragionamenti completi sono in `ragionamenti.csv`. Se la sessione cade, rilancia: il notebook riprende dai blocchi salvati, con gli stessi semi.

In [ ]:
# 1. Configurazione. Le celle 2–4 girano anche su CPU (preflight); l'inferenza parte solo con AVVIA_GPU = True.
MODALITA = "prova"          # "prova" (10 storie, Qwen3-8B acceso e spento) oppure "completo" (200 storie, 800 coppie)
MODELLI = ["qwen3_8b"]
# completo: ["qwen3_8b", "qwen3_32b", "phi4_reasoning"]; facoltativo in coda: "r1_llama_8b"
# Serve la sessione A100-SXM4-80GB: Qwen3-32B in bfloat16 non entra nella 40 GB.
AVVIA_GPU = False
SOLO_A100 = True            # ferma l'inferenza su GPU diverse da A100 (come nel Test M)
USA_DRIVE = True            # uscite in MyDrive/indetbench_reasoning/<modalità>/ (una sola copia)

# Codice e benchmark: repository pubblica, commit fissato. Il calcolo di p(UNKNOWN), W, soglia e report è quello
# delle parti precedenti (identico a c086e65); questo notebook aggiunge solo la generazione con ragionamento.
REPO = "Sva76/indet-bench"
REPO_COMMIT = "2aa1f6280f14d2c85c02923f9fce6e425678a293"
# Controllo su CPU dei 4 W del Test U2 (cella 4b), obbligatorio prima del completo. Una sola copia in MyDrive.
ARCHIVIO_TEST_U2 = "testU2_risultati.zip"
# Secrets di Colab (icona a chiave), con «Accesso al notebook» attivo per QUESTO notebook:
#   GH_TOKEN: facoltativo (la repository è pubblica); se presente, usato in sola lettura e mai stampato
#   HF_TOKEN: token Hugging Face (nessuno dei modelli proposti è gated; serve comunque per i limiti di download)

In [ ]:
# 2. Drive, repository al commit fissato, benchmark congelato. Nessuna GPU richiesta.
import hashlib, json, os, re, subprocess, sys, zipfile
from pathlib import Path

VERSIONE = "R-notebook-1"
assert MODALITA in ("prova", "completo"), MODALITA
SMOKE = MODALITA == "prova"
REPO_DIR = Path(os.environ.get("INDETBENCH_REPO_DIR", "/content/indet-bench"))

_SEGRETI = []

def nascondi(testo):
    """Toglie i token noti da qualsiasi testo che finisce a schermo o nei file."""
    testo = str(testo)
    for s in _SEGRETI:
        if s:
            testo = testo.replace(s, "***")
    return testo

def segreto(nome):
    valore = os.environ.get(nome)
    if not valore:
        try:
            from google.colab import userdata
            valore = userdata.get(nome)
        except Exception:
            valore = None
    if valore:
        _SEGRETI.append(valore)
    return valore

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {args[0]} fallito: {r.stderr.strip()[:300]}")
    return r.stdout.strip()

if not (REPO_DIR / ".git").exists():
    gh = segreto("GH_TOKEN")    # facoltativo: la repository è pubblica
    import base64
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git("init", "-q", str(REPO_DIR), cwd=REPO_DIR.parent)
    # Il token passa solo in una variabile d'ambiente del processo git (intestazione HTTP): non compare
    # negli argomenti, nell'URL, in .git/config o in FETCH_HEAD, e non viene stampato.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    cred = None
    if gh:
        cred = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
        _SEGRETI.append(cred)
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {cred}")
    r = subprocess.run(["git", "fetch", "-q", "--depth", "1", f"https://github.com/{REPO}.git", REPO_COMMIT],
                       cwd=REPO_DIR, capture_output=True, text=True, env=env)
    del env, cred
    if r.returncode:
        raise RuntimeError("Download della repository fallito: " + nascondi(r.stderr)[:300])
    git("checkout", "-q", "FETCH_HEAD")
    (REPO_DIR / ".git" / "FETCH_HEAD").unlink(missing_ok=True)
assert git("rev-parse", "HEAD") == REPO_COMMIT, "La copia locale non è al commit fissato"
assert not git("status", "--porcelain", "--untracked-files=no"), "La copia locale della repository è modificata"
sys.path.insert(0, str(REPO_DIR / "src"))

from indetbench import original_benchmark as ob, hf_runner as hr
from indetbench.metrics import PairScore, w_score

FROZEN = ob.load_frozen(REPO_DIR / "research_originals/step2/frozen/M_Original_4800_righe.json.gz")  # controlla SHA-256
TUTTE = ob.export_rows(FROZEN, "PN")          # 1600 righe B/C, 800 coppie, 200 storie, prompt invariati
GOLD = {r["row_id"]: r for r in FROZEN}

def storie_prova():
    # Protocollo parte B §6: 10 storie (V2001–V2010, le famiglie si alternano), comprendono V2001–V2003 del controllo
    ordine = list(dict.fromkeys(r["block_id"] for r in TUTTE))
    return set(ordine[:10])

def storie_ripetibilita():
    # Protocollo parte B §5.4: prime 20 storie (80 coppie) nel completo; nella prova le prime 2, solo per provare il codice
    ordine = list(dict.fromkeys(r["block_id"] for r in TUTTE))
    return set(ordine[:2] if SMOKE else ordine[:20])

RIGHE = [r for r in TUTTE if r["block_id"] in storie_prova()] if SMOKE else TUTTE
DATA_SHA = hashlib.sha256("\n".join(r["row_id"] + "\t" + r["prompt"] for r in RIGHE).encode()).hexdigest()
n_coppie = len({r["pair_id"] for r in RIGHE})
assert SMOKE or (len(RIGHE) == 1600 and n_coppie == 800)

ROOT = Path("/content/drive/MyDrive" if USA_DRIVE else "/content") / "indetbench_reasoning" / MODALITA
if USA_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT.mkdir(parents=True, exist_ok=True)
print(f"Modalità {MODALITA} | repository {REPO_COMMIT[:7]} | benchmark SHA-256 {ob.FROZEN_SHA256[:12]} (verificato)")
print(f"Righe {len(RIGHE)} | coppie {n_coppie} | storie {len({r['block_id'] for r in RIGHE})} | dati {DATA_SHA[:16]}")
print(f"Uscite in {ROOT}")
esempio = next(r for r in RIGHE if r["side"] == "C")
print(f"\n--- esempio C ({esempio['pair_id']}) ---\n{esempio['prompt']}")

In [ ]:
# 3. Modelli, revisioni e regole del Protocollo parte B v1, approvato da Simona il 15 novembre 2026 prima di qualsiasi
# dato (decisioni §7: modelli, campionamento degli autori, budget 4096, batch + sdpa, nessuna misura alternativa).
# Le revisioni sono commit completi di Hugging Face, letti dall'API il 15 nov; la cella 5 controlla che esistano.
MODEL_CONFIG = {
    "qwen3_8b": dict(id="Qwen/Qwen3-8B", sha="b968826d9c46dd6066d109eabc6255188de91218", key="qwen",
                     ruolo="PRINCIPALE: stessi pesi con ragionamento acceso e spento (controllo positivo delle parti precedenti)",
                     licenza="Apache-2.0", condizioni=("spento", "acceso"), batch=16, gpu_min_gb=40, disco_gb=17,
                     campionamento=dict(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0),
                     confronto="stessi pesi, ragionamento spento (rimisurato in questa pipeline; Test U2 W 0.975)"),
    "qwen3_32b": dict(id="Qwen/Qwen3-32B", sha="9216db5781bf21249d130ec9da846c4624c16137", key="qwen",
                      ruolo="secondario: stessi pesi con ragionamento acceso e spento", licenza="Apache-2.0",
                      condizioni=("spento", "acceso"), batch=4, gpu_min_gb=75, disco_gb=67,
                      campionamento=dict(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0),
                      confronto="stessi pesi, ragionamento spento (rimisurato in questa pipeline; parte A W 0.975)"),
    "phi4_reasoning": dict(id="microsoft/Phi-4-reasoning", sha="1de18ec97600877ce63dbf60c73b998da99f0195", key="other",
                           ruolo="descrittivo: stessa base di Phi-4, addestramento diverso", licenza="MIT",
                           condizioni=("acceso",), batch=16, gpu_min_gb=40, disco_gb=30,
                           campionamento=dict(temperature=0.8, top_p=0.95, top_k=50, min_p=0.0),
                           confronto="Phi-4 del 14 novembre (W 0.944), pesi diversi: solo descrittivo"),
    "r1_llama_8b": dict(id="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", sha="6a6f4aa4197940add57724a7707d069478df56b1",
                        key="other", ruolo="facoltativo, descrittivo: altra famiglia di ragionamento",
                        licenza="MIT; derivato da Llama-3.1-8B-Base, vale anche la Llama 3.1 Community License",
                        condizioni=("acceso",), batch=16, gpu_min_gb=40, disco_gb=17,
                        campionamento=dict(temperature=0.6, top_p=0.95, top_k=0, min_p=0.0),   # top_k 0 = nessun taglio
                        confronto="nessun confronto a modello fisso"),
}
# Batch: 16 come da protocollo; Qwen3-32B usa 4 perché con 16 sequenze da ~4300 token la cache (~18 GB) più i pesi
# (~66 GB) supera gli 80 GB della GPU. La composizione dei batch e il seme sono registrati.

BUDGET_RAGIONAMENTO = 4096          # §4: token massimi di ragionamento; oltre, chiusura forzata registrata
CHIUSURA_FORZATA = "\n</think>\n\n" # §4: testo aggiunto se il ragionamento non si chiude da solo
TOKEN_RISPOSTA = 16                 # §4: token generati dopo </think> e letti con parse_label
SEME = 20261108                     # §4: seme fisso; ogni batch usa SEME + indice del batch (ripresa identica)
SEMI_RIPETIBILITA = (20261109, 20261110)   # §5.4: due semi in più sulle prime 20 storie

# Regole fissate prima dei dati
MASSA_MEDIANA_MIN = 0.5             # §5.2 (dalla v4): massa mediana < 0.5 al primo token della risposta → NON MISURABILE
COPERTURA_MIN = 760                 # §5.2: coppie con B e C chiusi da soli; sotto, il risultato del modello è descrittivo
TOLL_PIPELINE = 0.01                # §5.1: la pipeline a batch, ragionamento spento, riproduce i 24 valori del Test U2
MASSA_BASSA, QUOTA_MASSA = 0.90, 0.05   # cautela del Test U2 (descrittiva)
for k in MODELLI:
    assert k in MODEL_CONFIG, f"Modello sconosciuto: {k}"
    hr.validate_revision(MODEL_CONFIG[k]["sha"])

# Controllo della pipeline: 24 valori di p(UNKNOWN) di Qwen3-8B sulle storie V2001–V2003 (set PN, B e C),
# dal punteggi.csv archiviato del Test U2 (9 nov 2026).
REF_QWEN_PN = [["PN|V2001|POS|prima|granted|B", 1.0816490064735618e-11, "NO"], ["PN|V2001|POS|ultima|granted|B", 1.3711326118936995e-06, "NO"], ["PN|V2001|POS|prima|granted|C", 5.905622102453376e-10, "YES"], ["PN|V2001|POS|ultima|granted|C", 0.9859082102775574, "UNKNOWN"], ["PN|V2001|NOT|prima|granted|B", 1.955604211900664e-08, "NO"], ["PN|V2001|NOT|ultima|granted|B", 0.0001234118099091, "NO"], ["PN|V2001|NOT|prima|granted|C", 6.826222431755016e-08, "YES"], ["PN|V2001|NOT|ultima|granted|C", 0.5621962547302246, "UNKNOWN"], ["PN|V2002|POS|prima|granted|B", 3.266230601955966e-13, "YES"], ["PN|V2002|POS|ultima|granted|B", 1.5428827106981485e-13, "YES"], ["PN|V2002|POS|prima|granted|C", 0.0116246016696095, "NO"], ["PN|V2002|POS|ultima|granted|C", 0.0002034515491686, "YES"], ["PN|V2002|NOT|prima|granted|B", 6.826094534062578e-08, "YES"], ["PN|V2002|NOT|ultima|granted|B", 1.5231051975206356e-08, "YES"], ["PN|V2002|NOT|prima|granted|C", 2.1415366063592955e-05, "NO"], ["PN|V2002|NOT|ultima|granted|C", 0.0008244285709224, "YES"], ["PN|V2003|POS|prima|granted|B", 1.1862326232403577e-08, "YES"], ["PN|V2003|POS|ultima|granted|B", 3.059368793856265e-07, "YES"], ["PN|V2003|POS|prima|granted|C", 0.0001233841758221, "YES"], ["PN|V2003|POS|ultima|granted|C", 0.676827073097229, "UNKNOWN"], ["PN|V2003|NOT|prima|granted|B", 1.0130904229299633e-05, "YES"], ["PN|V2003|NOT|ultima|granted|B", 0.0004306302289478, "YES"], ["PN|V2003|NOT|prima|granted|C", 0.0005002911202609, "YES"], ["PN|V2003|NOT|ultima|granted|C", 0.1642791032791137, "YES"]]
# Controllo CPU (cella 4b): punteggi.csv del Test U2 (SHA-256 sotto) e W attesi su PN, 800 coppie.
PUNTEGGI_U2_SHA256 = "9701dce0dc7516665bc4a4c52026e52a0f9b7a2ccc3df01b660be7d9406d31aa"
W_ATTESI_U2_PN = {"qwen": 0.975, "mistral": 0.7825, "olmo": 0.69375, "llama": 0.74625}
NOMI_U2 = {"qwen": "Qwen/Qwen3-8B", "mistral": "mistralai/Mistral-Nemo-Instruct-2407",
           "olmo": "allenai/OLMo-2-1124-7B-Instruct", "llama": "meta-llama/Llama-3.1-8B-Instruct"}

In [ ]:
# 4. Analisi (solo CPU). Il risultato principale e la soglia di competenza vengono da indetbench.original_benchmark.report
# al commit fissato; qui si aggiungono le baseline dalle uscite e i controlli sugli artefatti, tutti DESCRITTIVI.
import math
from collections import Counter, defaultdict

SEED_BOOT = 20261108
LABELS = ("YES", "NO", "UNKNOWN")

def encode(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode("utf-8")

def fingerprint(obj):
    return hashlib.sha256(encode(obj)).hexdigest()

def write_json(path, obj):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(encode(obj))
    os.replace(tmp, path)

def normalizzate(r):
    p = [float(r[f"p_{k.lower()}"]) for k in LABELS]
    s = max(sum(p), 1e-300)
    return [x / s for x in p]

NORMALIZZATA = "p_UNKNOWN normalizzata su YES+NO+UNKNOWN"
NOTA_NORMALIZZATA = ("La baseline «p(UNKNOWN) normalizzata» è p(UNKNOWN)/(p(YES)+p(NO)+p(UNKNOWN)): quando la massa "
                     "sulle tre risposte è circa 1 coincide con p(UNKNOWN), quindi la parità con essa è attesa e non è "
                     "un'anomalia del metodo. Resta nel confronto per trasparenza: se il suo W differisce da quello di "
                     "p(UNKNOWN), la differenza misura l'effetto della massa fuori dalle tre risposte. Non entra nella "
                     "segnalazione «una baseline eguaglia o supera p(UNKNOWN)», che usa solo le baseline indipendenti.")

MISURE = {   # punteggio più alto = più indeterminato
    "p_UNKNOWN (principale)": lambda r: float(r["score"]),
    NORMALIZZATA: lambda r: normalizzate(r)[2],
    "meno p massima normalizzata": lambda r: -max(normalizzate(r)),
    "entropia normalizzata": lambda r: -sum(x * math.log(max(x, 1e-300)) for x in normalizzate(r)),
    "risposta generata UNKNOWN (0/1)": lambda r: float(r["parsed"] == "UNKNOWN"),
    "lunghezza del prompt in token": lambda r: float(r["prompt_tokens"]),
    # Protocollo parte B §5.4: baseline nuove, indipendenti (entrano nella segnalazione «baseline eguaglia»)
    "ragionamento: lunghezza in token": lambda r: float(r.get("ragionamento_token") or 0),
    "ragionamento troncato (0/1)": lambda r: float(str(r.get("chiusura") or "naturale") != "naturale"),
}

def stima_W(rows, misura, boot, coppie_ok=None):
    pairs = [PairScore(r["pair_id"], r["block_id"], r["side"], misura(r)) for r in rows
             if coppie_ok is None or r["pair_id"] in coppie_ok]
    if not pairs:
        return dict(stima=None, ci95=None, coppie=0, storie=0)
    e = w_score(pairs, n_bootstrap=boot, seed=SEED_BOOT)
    ci = None if e.ci_lower is None else [e.ci_lower, e.ci_upper]
    return dict(stima=e.value, ci95=ci, coppie=e.n_pairs, storie=e.n_blocks)

def auroc(rows, misura):
    c = [misura(r) for r in rows if r["side"] == "C"]
    b = [misura(r) for r in rows if r["side"] == "B"]
    if not c or not b:
        return None
    vinte = sum((x > y) + 0.5 * (x == y) for x in c for y in b)
    return vinte / (len(c) * len(b))

def filtro_lunghezza(rows, lunghezze):
    # Regola del Test U2: B e C della stessa coppia devono avere lo stesso numero di token, altrimenti la coppia esce.
    per_coppia = defaultdict(set)
    for r in rows:
        per_coppia[r["pair_id"]].add(lunghezze[r["row_id"]])
    via = {p for p, s in per_coppia.items() if len(s) != 1}
    tenute = [r for r in rows if r["pair_id"] not in via]
    scarti = [dict(row_id=r["row_id"], pair_id=r["pair_id"], prompt_tokens=lunghezze[r["row_id"]])
              for r in rows if r["pair_id"] in via]
    return tenute, scarti

ATTRIBUTI = ("family", "variant", "position", "condition")

def distorsione_esclusioni(tutte, scarti):
    """Le coppie escluse sono distribuite come tutte le coppie? Tabella per attributo, solo descrittiva."""
    escluse = {s["pair_id"] for s in scarti}
    coppie = {r["pair_id"]: GOLD[r["row_id"]] for r in tutte if r["side"] == "B"}
    tab = {}
    for a in ATTRIBUTI:
        conta = defaultdict(lambda: [0, 0])
        for p, g in coppie.items():
            conta[g[a]][0] += 1
            conta[g[a]][1] += p in escluse
        tab[a if a != "condition" else "gold_B"] = {k: dict(coppie=n, escluse=e, quota=e / n) for k, (n, e) in sorted(conta.items())}
    storie = Counter(coppie[p]["story_id"] for p in escluse)
    quota = len(escluse) / max(len(coppie), 1)
    # concentrazione: un gruppo con almeno 5 escluse e quota più che doppia di quella complessiva
    concentrate = [f"{a}={k} ({v['escluse']}/{v['coppie']})" for a, d in tab.items() for k, v in d.items()
                   if v["escluse"] >= 5 and v["quota"] > 2 * quota]
    return dict(coppie_escluse=len(escluse), quota=quota, per_attributo=tab,
                storie_con_esclusioni=len(storie), max_per_storia=max(storie.values(), default=0),
                gruppi_concentrati=concentrate, elenco=sorted(escluse))

def analizza(nome, cfg, rows, scarti, smoke, extra=None, tutte=None):
    """rows: coppie tenute (analisi principale); tutte: tutte le righe misurate, comprese le coppie escluse."""
    boot = 0 if smoke else 2000
    rep = ob.report(rows, FROZEN, n_bootstrap=boot, seed=SEED_BOOT)
    coppie = rep["W_p_unknown"]["n_pairs"]
    if smoke:
        esito = "PROVA TECNICA (nessuna lettura scientifica)"
    elif coppie < COPERTURA_MIN:
        esito = f"DESCRITTIVO (solo {coppie} coppie valide, minimo {COPERTURA_MIN})"
    else:
        esito = rep["verdict"]
    per_coppia = defaultdict(dict)
    for r in rows:
        per_coppia[r["pair_id"]][r["side"]] = r
    attr = {p: GOLD[d["B"]["row_id"]] for p, d in per_coppia.items()}
    def gruppo(f):
        out = defaultdict(set)
        for p, g in attr.items():
            out[f(p, g)].add(p)
        return out
    controlli = {}
    for nome_g, f in (("famiglia", lambda p, g: g["family"]), ("variante", lambda p, g: g["variant"]),
                      ("posizione", lambda p, g: g["position"]),
                      ("famiglia x variante", lambda p, g: f"{g['family']}/{g['variant']}")):
        controlli[f"W per {nome_g}"] = {k: stima_W(rows, MISURE["p_UNKNOWN (principale)"], boot, v)
                                        for k, v in sorted(gruppo(f).items())}
    unk = {p: (d["B"]["parsed"] == "UNKNOWN", d["C"]["parsed"] == "UNKNOWN") for p, d in per_coppia.items()}
    parita = {"nessuna delle due risposte UNKNOWN": {p for p, u in unk.items() if not any(u)},
              "entrambe le risposte UNKNOWN": {p for p, u in unk.items() if all(u)},
              "C senza risposta UNKNOWN": {p for p, u in unk.items() if not u[1]}}
    controlli["W a parità di risposta generata"] = {k: stima_W(rows, MISURE["p_UNKNOWN (principale)"], boot, v)
                                                    for k, v in parita.items()}
    risposte = {s: Counter(r["parsed"] for r in rows if r["side"] == s) for s in ("B", "C")}
    massa_bassa = sum(float(r["mass_total"]) < MASSA_BASSA for r in rows) / max(len(rows), 1)
    return dict(
        modello=cfg["id"], revisione=cfg["sha"], ruolo=cfg["ruolo"], chiave=nome,
        classe="ESPLORATIVO: parte B (ragionamento), non è una validazione indipendente",
        esito=esito, report_indetbench=rep,
        baseline_dalle_uscite={k: dict(W=stima_W(rows, m, boot), AUROC=auroc(rows, m)) for k, m in MISURE.items()},
        controlli_artefatti=controlli,
        risposte_generate={s: dict(c) for s, c in risposte.items()},
        quota_UNKNOWN={s: risposte[s]["UNKNOWN"] / max(sum(risposte[s].values()), 1) for s in ("B", "C")},
        coppie_scartate_lunghezza=len({s["pair_id"] for s in scarti}),
        esclusioni=dict(
            distorsione=distorsione_esclusioni(tutte or rows, scarti),
            # sensibilità: stesso W calcolato anche sulle coppie escluse (misurate comunque); solo descrittivo
            W_sensibilita_tutte_le_coppie=(stima_W(tutte, MISURE["p_UNKNOWN (principale)"], boot)
                                           if tutte and scarti else None)),
        quota_massa_bassa=massa_bassa, cautela_massa=massa_bassa > QUOTA_MASSA,
        **(extra or {}))


def riga_riepilogo(e):
    rep = e["report_indetbench"]; g = rep["competence_gate"]; w = rep["W_p_unknown"]
    base = e["baseline_dalle_uscite"]
    fam = rep["W_by_family_descriptive"]
    return dict(modello=e["chiave"], classe=e["classe"].split(":")[0], esito=e["esito"],
                W=w["value"], IC95_basso=w["ci_lower"], IC95_alto=w["ci_upper"], coppie=w["n_pairs"],
                acc_B=g["accuracy_B"], acc_B_gold_NO=g["accuracy_B_by_gold"]["NO"],
                acc_B_gold_YES=g["accuracy_B_by_gold"]["YES"], soglia_superata=g["passed"],
                W_AND=fam["AND"], W_OR=fam["OR"], W_IMPLIES=fam["IMPLIES"],
                W_entropia=base["entropia normalizzata"]["W"]["stima"],
                W_meno_p_max=base["meno p massima normalizzata"]["W"]["stima"],
                W_risposta_UNKNOWN=base["risposta generata UNKNOWN (0/1)"]["W"]["stima"],
                W_lunghezza=base["lunghezza del prompt in token"]["W"]["stima"],
                UNK_B=e["quota_UNKNOWN"]["B"], UNK_C=e["quota_UNKNOWN"]["C"],
                scarti_lunghezza=e["coppie_scartate_lunghezza"], cautela_massa=e["cautela_massa"])

def _segnala(lista, cosa, prova, categoria_possibile):
    # Ogni anomalia resta «DA CLASSIFICARE»: la causa va documentata da chi legge, con prove e controlli.
    lista.append(dict(anomalia=cosa, prova=prova, categorie_possibili=categoria_possibile,
                      classificazione="DA CLASSIFICARE", causa_ipotizzata="", controlli_effettuati="",
                      cosa_resta_incerto=""))

def livelli(e):
    """Tre livelli separati (criterio di Simona, 14 nov 2026) e anomalie da classificare."""
    rep = e["report_indetbench"]; g = rep["competence_gate"]; w = rep["W_p_unknown"]
    base = e["baseline_dalle_uscite"]; ctrl = e["controlli_artefatti"]
    anomalie = []
    risposte = Counter(); [risposte.update(c) for c in e["risposte_generate"].values()]
    n = sum(risposte.values())
    if e["coppie_scartate_lunghezza"]:
        _segnala(anomalie, "coppie con B e C di lunghezza diversa in token, escluse",
                 f"{e['coppie_scartate_lunghezza']} coppie", "problema tecnico (tokenizzazione) / limite del protocollo")
    dist = e["esclusioni"]["distorsione"]
    if dist["gruppi_concentrati"]:
        _segnala(anomalie, "esclusioni per lunghezza concentrate in alcuni gruppi (possibile distorsione)",
                 "; ".join(dist["gruppi_concentrati"]), "limite del protocollo / problema tecnico (tokenizzazione)")
    sens = e["esclusioni"]["W_sensibilita_tutte_le_coppie"]
    if sens and w["value"] is not None and abs(sens["stima"] - w["value"]) >= 0.02:
        _segnala(anomalie, "W cambia di almeno 0.02 includendo le coppie escluse",
                 f"{w['value']:.4f} senza, {sens['stima']:.4f} con", "limite del protocollo")
    if risposte["ALTRO"] / max(n, 1) > 0.05:
        _segnala(anomalie, "risposte non leggibili come YES/NO/UNKNOWN", f"{risposte['ALTRO']}/{n}",
                 "comportamento del modello / problema tecnico (template, lettura della risposta)")
    if e["cautela_massa"]:
        _segnala(anomalie, "massa di probabilità YES+NO+UNKNOWN bassa al primo token",
                 f"{e['quota_massa_bassa']:.1%} delle righe < {MASSA_BASSA}",
                 "comportamento del modello (formato della risposta) / problema tecnico (template)")
    for w_ in rep["warnings"]:
        _segnala(anomalie, w_, "report di indetbench", "comportamento del modello")
    if not g["passed"]:
        _segnala(anomalie, "soglia di competenza sui B non superata", f"parti fallite: {g['failed_parts']}",
                 "comportamento del modello / limite del protocollo")
    fam = [v for v in rep["W_by_family_descriptive"].values() if v is not None]
    if fam and max(fam) - min(fam) >= 0.20:
        _segnala(anomalie, "W molto diverso fra famiglie (differenza ≥ 0.20, soglia descrittiva del Test M)",
                 json.dumps(rep["W_by_family_descriptive"]), "limite del protocollo / comportamento del modello")
    # Decisione di Simona (14 nov, prima dei dati del completo): la baseline «p(UNKNOWN) normalizzata» resta nel
    # confronto, ma non è una baseline indipendente: normalizzata = p(UNKNOWN) / (p(YES)+p(NO)+p(UNKNOWN)). Se la massa
    # sulle tre risposte è ≈ 1 coincide con p(UNKNOWN), quindi una parità con essa NON è un'anomalia del metodo.
    # Si segnala solo se ne differisce (|ΔW| ≥ 0.02): allora conta la massa fuori dalle tre risposte.
    norm = base[NORMALIZZATA]["W"]["stima"]
    if norm is not None and w["value"] is not None and abs(norm - w["value"]) >= 0.02:
        _segnala(anomalie, "p(UNKNOWN) normalizzata dà un W diverso da p(UNKNOWN): la massa fuori da YES/NO/UNKNOWN conta",
                 f"{norm:.3f} contro {w['value']:.3f}; quota righe con massa < {MASSA_BASSA}: {e['quota_massa_bassa']:.1%}",
                 "comportamento del modello / problema tecnico (template, formato della risposta)")
    migliori = {k: v["W"]["stima"] for k, v in base.items()
                if k not in ("p_UNKNOWN (principale)", NORMALIZZATA) and not k.startswith("lunghezza")
                and v["W"]["stima"] is not None}
    if migliori and w["value"] is not None and max(migliori.values()) >= w["value"]:
        k = max(migliori, key=migliori.get)
        _segnala(anomalie, "una baseline indipendente dalle uscite eguaglia o supera p(UNKNOWN)",
                 f"{k}: {migliori[k]:.3f} contro {w['value']:.3f}", "evidenza scientifica")
    al = e.get("allineamento_test_U2")
    if al and not al.get("ambiente_allineato", True):
        _segnala(anomalie, "Qwen3-8B non riproduce il Test U2", json.dumps(al), "problema tecnico (ambiente, versioni)")
    lung = base["lunghezza del prompt in token"]["W"]["stima"]
    return {
        "1. Funziona tecnicamente?": dict(
            revisione_verificata=True, template_e_etichette="controllati", coppie_valide=w["n_pairs"],
            coppie_scartate_lunghezza=e["coppie_scartate_lunghezza"],
            esclusioni_per_attributo=dist["per_attributo"], W_con_coppie_escluse=sens,
            controlli_tokenizer=e.get("controlli_tokenizer"), risposte_ALTRO=risposte["ALTRO"],
            quota_massa_bassa=e["quota_massa_bassa"], controllo_positivo_Qwen=al),
        "2. Misura ciò che intendiamo?": dict(
            soglia_competenza=dict(superata=g["passed"], acc_B=g["accuracy_B"], per_gold=g["accuracy_B_by_gold"]),
            W_per_famiglia=rep["W_by_family_descriptive"],
            W_per_variante={k: v["stima"] for k, v in ctrl["W per variante"].items()},
            W_per_posizione={k: v["stima"] for k, v in ctrl["W per posizione"].items()},
            W_a_parita_di_risposta={k: v["stima"] for k, v in ctrl["W a parità di risposta generata"].items()},
            W_lunghezza_prompt=lung),
        "3. Aggiunge rispetto alle baseline?": dict(
            W_p_UNKNOWN=w["value"], IC95=[w["ci_lower"], w["ci_upper"]],
            W_baseline={k: v["W"]["stima"] for k, v in base.items() if not k.startswith("p_UNKNOWN (")},
            nota=("confronto descrittivo; non c'è un test preregistrato di superiorità sulle baseline per i modelli nuovi. "
                  + NOTA_NORMALIZZATA)),
        "anomalie": anomalie,
    }
def W_per_storia(rows, misura, coppie_ok=None):
    """W per storia (media delle coppie, pareggi = mezzo punto), come w_score di indetbench."""
    per = defaultdict(dict)
    for r in rows:
        if coppie_ok is None or r["pair_id"] in coppie_ok:
            per[(r["block_id"], r["pair_id"])][r["side"]] = misura(r)
    storie = defaultdict(list)
    for (b, _), d in per.items():
        if "B" in d and "C" in d:
            storie[b].append((d["C"] > d["B"]) + 0.5 * (d["C"] == d["B"]))
    return {b: sum(v) / len(v) for b, v in storie.items()}

def delta_W_appaiato(rows_a, rows_b, boot, misura=None):
    """HR2: ΔW = W(a) − W(b), appaiato per storia, IC95 bootstrap sulle storie (seme SEED_BOOT)."""
    import numpy as np
    misura = misura or MISURE["p_UNKNOWN (principale)"]
    a, b = W_per_storia(rows_a, misura), W_per_storia(rows_b, misura)
    comuni = sorted(set(a) & set(b))
    if not comuni:
        return dict(stima=None, ci95=None, storie=0)
    d = np.array([a[s] - b[s] for s in comuni])
    out = dict(stima=float(d.mean()), W_a=float(np.mean([a[s] for s in comuni])),
               W_b=float(np.mean([b[s] for s in comuni])), storie=len(comuni), ci95=None, lettura=None)
    if boot:
        rng = np.random.default_rng(SEED_BOOT)
        bs = d[rng.integers(0, len(d), (boot, len(d)))].mean(1)
        lo, hi = (float(x) for x in np.quantile(bs, [0.025, 0.975]))
        out["ci95"] = [lo, hi]
        out["lettura"] = ("il ragionamento cambia W (IC95 esclude 0)" if lo > 0 or hi < 0 else
                          "nessun cambiamento dimostrato (IC95 comprende 0)")
    return out

print("Funzioni di analisi pronte.")

In [ ]:
# 4b. Controllo su CPU (obbligatorio prima del completo): l'analisi di questo notebook, applicata ai punteggi
# archiviati del Test U2, deve ridare esattamente i 4 W pubblicati su PN e i conteggi della soglia di competenza.
QWEN_U2_PN = {}   # row_id → (p_UNKNOWN, risposta) di Qwen3-8B nel Test U2, per il confronto descrittivo

def controllo_cpu_U2():
    import io
    import pandas as pd
    percorso = Path("/content/drive/MyDrive") / ARCHIVIO_TEST_U2 if USA_DRIVE else Path(ARCHIVIO_TEST_U2)
    if not percorso.is_file():
        return dict(eseguito=False, superato=False, motivo=f"archivio non trovato: {percorso}")
    with zipfile.ZipFile(percorso) as z:
        dati = z.read("punteggi.csv")
    sha = hashlib.sha256(dati).hexdigest()
    if sha != PUNTEGGI_U2_SHA256:
        return dict(eseguito=False, superato=False, motivo=f"punteggi.csv diverso dall'archiviato (SHA-256 {sha[:12]})")
    t = pd.read_csv(io.BytesIO(dati), keep_default_na=False)
    t = t[(t.set == "PN") & t.kind.isin(["B", "C"])]
    QWEN_U2_PN.update({r.row_id: (float(r.p_UNKNOWN), r.parsed) for r in t[t.modello == "qwen"].itertuples()})
    out = {}
    for key, d in t.groupby("modello"):
        rows = [dict(row_id=r.row_id, pair_id="PN|" + r.pair_key, block_id=r.story_id, side=r.kind,
                     score=r.p_UNKNOWN, p_yes=r.p_YES, p_no=r.p_NO, p_unknown=r.p_UNKNOWN, mass_total=r.mass_total,
                     generated=r.generated, parsed=r.parsed, prompt_tokens=r.prompt_tokens,
                     model_id=NOMI_U2[key], model_revision="archivio Test U2") for r in d.itertuples()]
        assert {r["row_id"] for r in rows} == {r["row_id"] for r in TUTTE}, "righe diverse dal benchmark congelato"
        tenute, scarti = filtro_lunghezza(rows, {r["row_id"]: r["prompt_tokens"] for r in rows})
        e = analizza(key, dict(id=NOMI_U2[key], sha="", ruolo="archivio"), tenute, scarti, False, tutte=rows)
        g = e["report_indetbench"]["competence_gate"]
        conteggi = {lab: (round(g["accuracy_B_by_gold"][lab] * g["n_B_by_gold"][lab]), g["n_B_by_gold"][lab])
                    for lab in ("NO", "YES")}
        w = e["report_indetbench"]["W_p_unknown"]
        ok = (abs(w["value"] - W_ATTESI_U2_PN[key]) < 1e-12 and w["n_pairs"] == 800 and
              conteggi == {k: tuple(v) for k, v in ob.REFERENCE_B_ACCURACY_BY_GOLD_PS[NOMI_U2[key]].items()} and
              e["esito"] == "CONFIRMED")
        out[key] = dict(W=w["value"], IC95=[w["ci_lower"], w["ci_upper"]], W_atteso=W_ATTESI_U2_PN[key],
                        conteggi_B_per_gold=conteggi, esito=e["esito"], superato=ok)
        print(f"  {key}: W {w['value']:.5f} (atteso {W_ATTESI_U2_PN[key]}) | {e['esito']} | "
              f"B corretti per gold {conteggi} | {'OK' if ok else 'DIVERSO'}")
    return dict(eseguito=True, superato=len(out) == 4 and all(v["superato"] for v in out.values()),
                sha256_punteggi=sha, modelli=out, repo_commit=REPO_COMMIT, versione=VERSIONE)

CONTROLLO_CPU = controllo_cpu_U2()
write_json(ROOT / "controllo_cpu_testU2.json", CONTROLLO_CPU)
print("Controllo CPU dei 4 W del Test U2:", "SUPERATO" if CONTROLLO_CPU["superato"] else
      f"NON SUPERATO ({CONTROLLO_CPU.get('motivo', 'valori diversi')})")
if not CONTROLLO_CPU["superato"] and not SMOKE:
    print("Il completo non partirà finché questo controllo non è superato.")

In [ ]:
# 5. Funzioni per la GPU: caricamento alla revisione fissata, controlli, generazione con e senza ragionamento.
# Misura (Protocollo parte B §2): p(UNKNOWN) = somma delle 6 varianti di UNKNOWN (hr.token_label_ids, nessuna
# rinormalizzazione) al primo token della risposta dopo </think>, cioè il primo token generato dopo </think> che non sia
# solo spazio o a capo. La risposta dopo </think> è generata in modo greedy (come nel Test U2); il campionamento
# riguarda solo il ragionamento. Senza ragionamento la misura è quella del Test U2: primo token generato, greedy, 8 token.
import csv, gc, platform, time

BLOCCO = 32   # righe per file di checkpoint (multiplo dei batch)

def hf_token():
    return segreto("HF_TOKEN")

def famiglia_gpu():
    import torch
    nome = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return nome, ("A100" if "A100" in nome else nome)

def ambiente():
    import torch, transformers
    nome, fam = famiglia_gpu()
    return dict(python=platform.python_version(), torch=str(torch.__version__),
                transformers=transformers.__version__, gpu=nome, famiglia_gpu=fam)

def memoria_gpu_gb():
    import torch
    return torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0

def verifica_revisione(cfg, token):
    from huggingface_hub import HfApi
    info = HfApi().model_info(cfg["id"], revision=cfg["sha"], token=token)
    if info.sha != cfg["sha"]:
        raise RuntimeError(f"{cfg['id']}: la revisione {cfg['sha']} non corrisponde ({info.sha})")

def carica_tokenizer(cfg, token):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(cfg["id"], revision=cfg["sha"], token=token, trust_remote_code=False)
    tok.padding_side = "left"
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    tok.model_input_names = ["input_ids", "attention_mask"]
    return tok

def carica_modello(cfg, token):
    import torch, transformers
    from transformers import AutoModelForCausalLM
    ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    dtype_arg = "dtype" if ver >= (4, 56) else "torch_dtype"
    return AutoModelForCausalLM.from_pretrained(
        cfg["id"], revision=cfg["sha"], token=token, device_map={"": 0}, attn_implementation="sdpa",
        trust_remote_code=False, use_safetensors=True, low_cpu_mem_usage=True, **{dtype_arg: torch.bfloat16}).eval()

def libera_cache(cfg):
    """Cancella dalla cache locale di Colab (non da Drive) i pesi del modello appena usato."""
    try:
        from huggingface_hub import scan_cache_dir
        info = scan_cache_dir()
        hashes = [rev.commit_hash for repo in info.repos if repo.repo_id == cfg["id"] for rev in repo.revisions]
        if hashes:
            info.delete_revisions(*hashes).execute()
            print(f"  cache locale di {cfg['id']} cancellata")
    except Exception as err:
        print("  ATTENZIONE: cache non cancellata:", nascondi(err)[:200])

def testo_al_modello(tok, cfg, prompt, condizione):
    """Solo il messaggio utente. Qwen3: enable_thinking acceso o spento. Altri modelli: il template così com'è
    (Phi-4-reasoning vi inserisce il proprio messaggio di sistema; dichiarato nel protocollo §3)."""
    if cfg["key"] == "qwen":
        return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=(condizione == "acceso"))
    return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)

def controlla_template(tok, cfg, condizione):
    t = tok.chat_template
    if isinstance(t, dict):
        t = t.get("default")
    if not t:
        return "il tokenizer non ha un chat template"
    if "strftime_now" in t:
        return "il chat template inserisce la data del giorno (il prompt cambierebbe con la data di lancio)"
    a = testo_al_modello(tok, cfg, "Question: test?", condizione)
    if a != testo_al_modello(tok, cfg, "Question: test?", condizione) or "Question: test?" not in a:
        return "il chat template non è deterministico o non contiene il messaggio"
    if condizione == "acceso" and len(tok.encode("</think>", add_special_tokens=False)) != 1:
        return "</think> non è un token singolo nel tokenizer: la fine del ragionamento non è identificabile"
    return None

def controlli_tokenizer(tok, cfg, rows, condizione):
    """Il prompt congelato compare intatto nel testo passato al modello; impronta dei testi."""
    testi = [testo_al_modello(tok, cfg, r["prompt"], condizione) for r in rows]
    mancanti = [r["row_id"] for r, t in zip(rows, testi) if r["prompt"] not in t]
    return dict(prompt_controllati=len(rows), prompt_non_intatti=mancanti[:20], n_prompt_non_intatti=len(mancanti),
                superato=not mancanti, condizione=condizione,
                sha256_testi_al_modello=hashlib.sha256("\n".join(testi).encode()).hexdigest())

def lunghezze(tok, cfg, rows, condizione):
    return {r["row_id"]: len(tok(testo_al_modello(tok, cfg, r["prompt"], condizione),
                                 add_special_tokens=False)["input_ids"]) for r in rows}

def _pad_sinistra(seqs, pad_id, device):
    import torch
    n = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), n), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), n), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, n - len(s):] = torch.tensor(s, dtype=torch.long)
        mask[i, n - len(s):] = 1
    return ids.to(device), mask.to(device)

def _etichette(dist, ids):
    return {lab: float(dist[ids[lab]].sum()) for lab in LABELS}

def _riga(r, probs, risposta, prompt_tokens, cfg, **extra):
    return dict(row_id=r["row_id"], pair_id=r["pair_id"], block_id=r["block_id"], side=r["side"],
                score=probs["UNKNOWN"], p_yes=probs["YES"], p_no=probs["NO"], p_unknown=probs["UNKNOWN"],
                mass_total=sum(probs.values()), generated=risposta, parsed=hr.parse_label(risposta),
                prompt_tokens=prompt_tokens, model_id=cfg["id"], model_revision=cfg["sha"], **extra)

def batch_spento(model, tok, cfg, ids, rows):
    """Ragionamento spento, misura del Test U2 (primo token generato, greedy, 8 token), ma a batch con sdpa."""
    import torch
    pad = tok.pad_token_id
    seqs = [tok(testo_al_modello(tok, cfg, r["prompt"], "spento"), add_special_tokens=False)["input_ids"] for r in rows]
    x, m = _pad_sinistra(seqs, pad, model.device)
    # parametri passati come argomenti di generate: hanno sempre la precedenza sulla generation_config del modello
    with torch.inference_mode():
        out = model.generate(input_ids=x, attention_mask=m, do_sample=False, max_new_tokens=8, pad_token_id=pad,
                             use_cache=True, return_dict_in_generate=True, output_logits=True)
    dist = out.logits[0].float().softmax(-1)
    if not bool(torch.isfinite(out.logits[0]).all()):
        raise ValueError("logit non finiti al primo passo")
    res = []
    for i, r in enumerate(rows):
        nuovi = out.sequences[i, x.shape[1]:].tolist()
        res.append(_riga(r, _etichette(dist[i], ids), tok.decode(nuovi, skip_special_tokens=True), len(seqs[i]), cfg,
                         condizione="spento", ragionamento_token=0, chiusura="", posizione_risposta=0))
    return res

def batch_acceso(model, tok, cfg, ids, rows, seme, indice):
    """Ragionamento acceso: (1) ragionamento campionato fino a </think> o al budget; (2) risposta greedy dopo </think>."""
    import torch
    pad = tok.pad_token_id
    fine = tok.encode("</think>", add_special_tokens=False)[0]
    eos = model.generation_config.eos_token_id
    eos = [eos] if isinstance(eos, int) else list(eos or [])
    if tok.eos_token_id is not None and tok.eos_token_id not in eos:
        eos.append(tok.eos_token_id)
    seqs = [tok(testo_al_modello(tok, cfg, r["prompt"], "acceso"), add_special_tokens=False)["input_ids"] for r in rows]
    x, m = _pad_sinistra(seqs, pad, model.device)
    c = cfg["campionamento"]
    torch.manual_seed(seme + indice)
    t0 = time.time()
    with torch.inference_mode():
        out1 = model.generate(input_ids=x, attention_mask=m, do_sample=True, temperature=c["temperature"],
                              top_p=c["top_p"], top_k=c["top_k"], min_p=c["min_p"] or None,
                              max_new_tokens=BUDGET_RAGIONAMENTO, pad_token_id=pad, eos_token_id=[fine] + eos,
                              use_cache=True)
    chiusa = tok.encode(CHIUSURA_FORZATA, add_special_tokens=False)
    prefissi, info = [], []
    for i in range(len(rows)):
        nuovi = out1[i, x.shape[1]:].tolist()
        stop = next((k for k, t in enumerate(nuovi) if t == fine or t in eos), None)
        if stop is not None and nuovi[stop] == fine:
            ragion, modo, coda = nuovi[:stop], "naturale", [fine]
        elif stop is not None:
            ragion, modo, coda = nuovi[:stop], "fine senza </think>", chiusa
        else:
            ragion, modo, coda = nuovi, "budget", chiusa
        prefissi.append(seqs[i] + ragion + coda)
        info.append((ragion, modo))
    x2, m2 = _pad_sinistra(prefissi, pad, model.device)
    with torch.inference_mode():
        out2 = model.generate(input_ids=x2, attention_mask=m2, do_sample=False, max_new_tokens=TOKEN_RISPOSTA,
                              pad_token_id=pad, eos_token_id=eos, use_cache=True,
                              return_dict_in_generate=True, output_logits=True)
    secondi = time.time() - t0
    res = []
    for i, r in enumerate(rows):
        nuovi = out2.sequences[i, x2.shape[1]:].tolist()
        k = next((j for j, t in enumerate(nuovi) if t not in eos and t != pad and tok.decode([t]).strip()), None)
        if k is None:
            probs = {lab: 0.0 for lab in LABELS}
        else:
            lg = out2.logits[k][i].float()
            if not bool(torch.isfinite(lg).all()):
                raise ValueError(f"logit non finiti: {r['row_id']}")
            probs = _etichette(lg.softmax(-1), ids)
        ragion, modo = info[i]
        risposta = tok.decode(nuovi, skip_special_tokens=True)
        res.append(_riga(r, probs, risposta, len(seqs[i]), cfg, condizione="acceso", ragionamento_token=len(ragion),
                         chiusura=modo, posizione_risposta=-1 if k is None else k, seme=seme, batch=indice,
                         secondi_batch=round(secondi, 2),
                         ragionamento=tok.decode(ragion, skip_special_tokens=False)))
    return res

def con_checkpoint(cartella, rows, calcola, batch, dopo_blocco=None):
    """Blocchi di BLOCCO righe salvati su Drive; dentro un blocco, batch fissi. Il seme dipende dall'indice globale
    del batch, quindi una ripresa ridà gli stessi batch con gli stessi semi."""
    cartella.mkdir(parents=True, exist_ok=True)
    out = []
    for i in range(0, len(rows), BLOCCO):
        parte_rows = rows[i:i + BLOCCO]
        path = cartella / f"{i // BLOCCO:04d}.json"
        if path.exists():
            parte = json.loads(path.read_text())
        else:
            parte = []
            for j in range(0, len(parte_rows), batch):
                parte += calcola(parte_rows[j:j + batch], (i + j) // batch)
            write_json(path, parte)
        if [r["row_id"] for r in parte] != [r["row_id"] for r in parte_rows]:
            raise RuntimeError(f"Checkpoint incoerente: {path}")
        out.extend(parte)
        print(f"  {len(out)}/{len(rows)}", flush=True)
        if dopo_blocco:
            motivo = dopo_blocco(out)
            if motivo:
                return out, motivo
    return out, None

def scrivi_csv(path, rows):
    tmp = path.with_suffix(".csv.tmp")
    campi = list(dict.fromkeys(k for r in rows for k in r)) or ["row_id"]
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campi, restval="")
        w.writeheader()
        w.writerows(rows)
    os.replace(tmp, path)

def esporta(cartella, esito, tabelle):
    write_json(cartella / "esito.json", esito)
    nomi = ["esito.json"]
    for nome, rows in tabelle.items():
        scrivi_csv(cartella / f"{nome}.csv", rows)
        nomi.append(f"{nome}.csv")
    archivio = cartella / f"{cartella.name}.zip"
    tmp = archivio.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for n in nomi:
            z.write(cartella / n, n)
    os.replace(tmp, archivio)
    return archivio

def mediana(v):
    import statistics
    v = list(v)
    return statistics.median(v) if v else None

def controllo_pipeline(model, tok, cfg, ids):
    """§5.1: pipeline a batch + sdpa, ragionamento spento, contro i 24 valori del Test U2 (tolleranza 0.01, 24/24)."""
    ref = {r: (v, p) for r, v, p in REF_QWEN_PN}
    righe = [r for r in TUTTE if r["row_id"] in ref]
    out = []
    for j in range(0, len(righe), cfg["batch"]):
        out += batch_spento(model, tok, cfg, ids, righe[j:j + cfg["batch"]])
    delta = max(abs(float(o["score"]) - ref[o["row_id"]][0]) for o in out)
    uguali = sum(o["parsed"] == ref[o["row_id"]][1] for o in out)
    return dict(n=len(out), scarto_max=delta, risposte_uguali=uguali, tolleranza=TOLL_PIPELINE,
                superato=delta <= TOLL_PIPELINE and uguali == len(out))

def non_chiusi_per_attributo(tenute):
    out = {}
    for a in ATTRIBUTI:
        tot, nc = Counter(), Counter()
        for o in tenute:
            k = f"{GOLD[o['row_id']][a]}/{o['side']}"
            tot[k] += 1
            nc[k] += o["chiusura"] != "naturale"
        out[a] = {k: dict(non_chiusi=nc[k], righe=tot[k]) for k in sorted(tot)}
    return out

def analizza_condizione(nome, cfg, cond, raw, scarti, extra):
    escluse = {s["pair_id"] for s in scarti}
    tenute = [o for o in raw if o["pair_id"] not in escluse]
    e = analizza(nome, cfg, tenute, scarti, SMOKE, dict(extra, condizione=cond), tutte=raw)
    boot = 0 if SMOKE else 2000
    # regola sulla massa (§5.2) sulle righe tenute, come nelle parti precedenti; la mediana su tutte le righe grezze
    # (comprese le coppie escluse per lunghezza) è registrata solo come diagnostica
    e["massa_mediana_tutte"] = mediana(float(o["mass_total"]) for o in tenute)
    e["massa_mediana_righe_grezze_diagnostica"] = mediana(float(o["mass_total"]) for o in raw)
    if cond == "acceso":
        chiusure = Counter(o["chiusura"] for o in tenute)
        per_coppia = defaultdict(set)
        for o in tenute:
            per_coppia[o["pair_id"]].add(o["chiusura"])
        chiuse = {p for p, s in per_coppia.items() if s == {"naturale"}}
        e["ragionamento"] = dict(
            chiusure=dict(chiusure), coppie_chiuse_da_sole=len(chiuse),
            token_mediana={s: mediana(int(o["ragionamento_token"]) for o in tenute if o["side"] == s) for s in "BC"},
            token_massimo=max(int(o["ragionamento_token"]) for o in tenute),
            W_sensibilita_solo_coppie_chiuse=stima_W(tenute, MISURE["p_UNKNOWN (principale)"], boot, chiuse),
            posizione_risposta=dict(Counter(int(o["posizione_risposta"]) for o in tenute)),
            # baseline descrittiva: ragionamenti non chiusi da soli per attributo del benchmark e lato
            non_chiusi_per_attributo=non_chiusi_per_attributo(tenute))
        if not SMOKE and len(chiuse) < COPERTURA_MIN:
            e["esito_del_codice_prima_della_regola_copertura"] = e["esito"]
            e["esito"] = (f"DESCRITTIVO: solo {len(chiuse)} coppie con ragionamento chiuso da solo "
                          f"(minimo {COPERTURA_MIN}, protocollo §5.2)")
    if e["massa_mediana_tutte"] is not None and e["massa_mediana_tutte"] < MASSA_MEDIANA_MIN:
        e["esito_del_codice_prima_della_regola_massa"] = e["esito"]
        e["esito"] = (f"NON MISURABILE: massa mediana sulle etichette {e['massa_mediana_tutte']:.3g} "
                      f"< {MASSA_MEDIANA_MIN} al primo token della risposta (regola tecnica)")
    e["livelli"] = livelli(e)
    return e, tenute

def valuta_modello(nome, token):
    import torch
    cfg = MODEL_CONFIG[nome]
    if memoria_gpu_gb() < cfg["gpu_min_gb"]:
        raise RuntimeError(f"{nome}: servono almeno {cfg['gpu_min_gb']} GB di memoria GPU, "
                           f"questa ne ha {memoria_gpu_gb():.0f} (usa la sessione A100-SXM4-80GB)")
    import shutil
    from huggingface_hub.constants import HF_HUB_CACHE
    Path(HF_HUB_CACHE).mkdir(parents=True, exist_ok=True)
    gia = any(Path(HF_HUB_CACHE).glob(f"models--{cfg['id'].replace('/', '--')}/snapshots/{cfg['sha']}"))
    if not gia and shutil.disk_usage(HF_HUB_CACHE).free / 1e9 < cfg["disco_gb"] + 5:
        raise RuntimeError(f"{nome}: servono circa {cfg['disco_gb']} GB di disco per i pesi")
    manifest = dict(versione=VERSIONE, modello=cfg["id"], revisione=cfg["sha"], dtype="bfloat16", attention="sdpa",
                    batch=cfg["batch"], budget_ragionamento=BUDGET_RAGIONAMENTO, token_risposta=TOKEN_RISPOSTA,
                    chiusura_forzata=CHIUSURA_FORZATA, campionamento=cfg["campionamento"], seme=SEME,
                    semi_ripetibilita=list(SEMI_RIPETIBILITA), condizioni=list(cfg["condizioni"]),
                    repo_commit=REPO_COMMIT, benchmark_sha256=ob.FROZEN_SHA256, dati=DATA_SHA, modalita=MODALITA,
                    **ambiente())
    cartella = ROOT / f"{nome}-{fingerprint(dict(manifest, gpu=None))[:12]}"
    cartella.mkdir(parents=True, exist_ok=True)
    if (cartella / f"{cartella.name}.zip").is_file():
        print(f"{nome}: già completato ({cartella.name})")
        return cartella
    write_json(cartella / "manifest.json", manifest)
    def fermo(motivo, **extra):
        esito = dict(manifest, chiave=nome, ruolo=cfg["ruolo"], esito=f"NON MISURABILE: {motivo}", **extra)
        print(f"  {nome}: NON MISURABILE, {motivo}")
        esporta(cartella, esito, {})
        return cartella
    verifica_revisione(cfg, token)
    tok = carica_tokenizer(cfg, token)
    for cond in cfg["condizioni"]:
        motivo = controlla_template(tok, cfg, cond)
        if motivo:
            return fermo(f"{motivo} (condizione {cond})")
    try:
        ids = hr.token_label_ids(tok)
    except ValueError as err:
        return fermo(f"etichette non separabili nel tokenizer ({err})")
    ctok = {c: controlli_tokenizer(tok, cfg, RIGHE, c) for c in cfg["condizioni"]}
    if not all(v["superato"] for v in ctok.values()):
        return fermo("prompt congelato non intatto nel testo al modello", controlli_tokenizer=ctok)
    lung = {c: lunghezze(tok, cfg, RIGHE, c) for c in cfg["condizioni"]}
    scarti = {c: filtro_lunghezza(RIGHE, lung[c])[1] for c in cfg["condizioni"]}
    print(f"  {nome}: etichette {ids} | coppie escluse per lunghezza "
          f"{ {c: len({s['pair_id'] for s in v}) for c, v in scarti.items()} }")
    controllo_etich = {r for r, _, _ in REF_QWEN_PN}   # V2001–V2003, 24 righe, all'inizio dell'ordine
    model = carica_modello(cfg, token)
    extra = dict(label_ids=ids, manifest=manifest, controlli_tokenizer=ctok,
                 generation_config_del_modello=model.generation_config.to_diff_dict())
    esiti, grezzi, tempi, fermati = {}, {}, {}, {}
    try:
        if nome == "qwen3_8b":
            cp = controllo_pipeline(model, tok, cfg, ids)
            extra["controllo_pipeline"] = cp
            print(f"  Controllo pipeline (spento, batch {cfg['batch']}, sdpa): scarto max {cp['scarto_max']:.3g} "
                  f"(tolleranza {TOLL_PIPELINE}), risposte uguali {cp['risposte_uguali']}/{cp['n']}")
            if not cp["superato"] and not SMOKE:
                raise RuntimeError("Controllo della pipeline non superato (§5.1): il completo non parte.")
        for cond in cfg["condizioni"]:
            t0 = time.time()
            if cond == "spento":
                calcola = lambda rr, i: batch_spento(model, tok, cfg, ids, rr)
                controllo = None
            else:
                calcola = lambda rr, i: batch_acceso(model, tok, cfg, ids, rr, SEME, i)
                def controllo(fatti):
                    if "copertura_etichette_24_righe" in extra:
                        return None
                    masse = [float(o["mass_total"]) for o in fatti if o["row_id"] in controllo_etich]
                    if len(masse) < len(controllo_etich):
                        return None
                    med = mediana(masse)
                    extra["copertura_etichette_24_righe"] = dict(massa_mediana=med, righe=len(masse),
                                                                 superato=med >= MASSA_MEDIANA_MIN)
                    print(f"  massa mediana sulle etichette (24 righe di controllo): {med:.3g}")
                    return None if med >= MASSA_MEDIANA_MIN else f"massa mediana {med:.3g} < {MASSA_MEDIANA_MIN} sulle 24 righe di controllo"
            raw, motivo = con_checkpoint(cartella / f"checkpoint-{cond}-{DATA_SHA[:12]}", RIGHE, calcola,
                                         cfg["batch"], controllo)
            tempi[cond] = round(time.time() - t0, 1)
            if motivo:   # la condizione accesa è NON MISURABILE; la condizione spenta resta analizzata
                fermati[cond] = dict(esito=f"NON MISURABILE: {motivo} (regola tecnica, protocollo §5.2)",
                                     copertura_etichette_24_righe=extra.get("copertura_etichette_24_righe"),
                                     righe_misurate_prima_dello_stop=len(raw))
                print(f"  {nome} ({cond}): {fermati[cond]['esito']}")
                continue
            for o in raw:
                if int(o["prompt_tokens"]) != lung[cond][o["row_id"]]:
                    raise RuntimeError(f"Lunghezza diversa fra controllo e inferenza: {o['row_id']}")
            grezzi[cond] = raw
        # §5.4 ripetibilità: due semi in più sulle prime 20 storie (2 nella prova), solo ragionamento acceso
        rip_rows = [r for r in RIGHE if r["block_id"] in storie_ripetibilita()]
        ripet = {}
        for s in (SEMI_RIPETIBILITA if "acceso" in grezzi else ()):
            raw_s, _ = con_checkpoint(cartella / f"ripetibilita-{s}-{DATA_SHA[:12]}", rip_rows,
                                      lambda rr, i, s=s: batch_acceso(model, tok, cfg, ids, rr, s, i), cfg["batch"])
            ripet[s] = raw_s
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        libera_cache(cfg)
    extra["secondi_per_condizione"] = tempi
    righe_out = []
    for cond, raw in grezzi.items():
        e, tenute = analizza_condizione(nome, cfg, cond, raw, scarti[cond], dict(extra, controlli_tokenizer=ctok[cond]))
        esiti[cond] = e
        righe_out += [dict(o, coppia_esclusa_lunghezza=o["pair_id"] in {s["pair_id"] for s in scarti[cond]}) for o in raw]
    boot = 0 if SMOKE else 2000
    sintesi = dict(manifest, chiave=nome, ruolo=cfg["ruolo"], confronto=cfg["confronto"], licenza=cfg["licenza"],
                   condizioni={c: dict(esito=e["esito"], W=e["report_indetbench"]["W_p_unknown"]["value"])
                               for c, e in esiti.items()},
                   identita=dict(dati_sha256=DATA_SHA, benchmark_sha256=ob.FROZEN_SHA256), **{k: v for k, v in extra.items()
                                                                                             if k != "manifest"})
    if "acceso" in grezzi and "spento" in grezzi:   # HR2: stessi pesi, ΔW acceso − spento appaiato per storia
        on, off = esiti["acceso"], esiti["spento"]
        fuori = {x["pair_id"] for v in scarti.values() for x in v}   # coppie escluse per lunghezza in una delle due
        hr2 = delta_W_appaiato(
            [o for o in grezzi["acceso"] if o["pair_id"] not in fuori],
            [o for o in grezzi["spento"] if o["pair_id"] not in fuori], boot)
        # HR2 si legge solo se entrambe le condizioni sono misurabili e la copertura (§5.2) è raggiunta
        motivi = [f"condizione {c}: {e['esito']}" for c, e in (("acceso", on), ("spento", off))
                  if str(e["esito"]).startswith(("NON MISURABILE", "DESCRITTIVO", "PROVA", "NOT EVALUABLE",
                                                     "DESCRIPTIVE", "NO INTERVAL"))]
        if SMOKE:
            motivi.append("modalità prova")
        hr2["lettura"] = "VALUTABILE (esplorativa)" if not motivi else "SOLO DESCRITTIVA: " + "; ".join(motivi)
        sintesi["HR2_delta_W_acceso_meno_spento"] = hr2
        cambi = {s: Counter() for s in "BC"}
        spento = {o["row_id"]: o["parsed"] for o in grezzi["spento"]}
        for o in grezzi["acceso"]:
            cambi[o["side"]][f"{spento[o['row_id']]} → {o['parsed']}"] += 1
        sintesi["risposte_spento_acceso"] = {s: dict(c) for s, c in cambi.items()}
    if nome == "qwen3_8b" and "spento" in grezzi and QWEN_U2_PN:   # descrittivo: spento (batch, sdpa) contro Test U2
        d = [abs(float(o["score"]) - QWEN_U2_PN[o["row_id"]][0]) for o in grezzi["spento"]]
        sintesi["spento_contro_test_U2"] = dict(righe=len(d), scarto_max=max(d),
                                                risposte_uguali=sum(o["parsed"] == QWEN_U2_PN[o["row_id"]][1]
                                                                    for o in grezzi["spento"]))
    if ripet and "acceso" in grezzi:
        rip_ids = {r["row_id"] for r in rip_rows}
        base_rows = [o for o in grezzi["acceso"] if o["row_id"] in rip_ids]
        risposte0 = {o["row_id"]: o["parsed"] for o in base_rows}
        sintesi["ripetibilita"] = dict(storie=len(storie_ripetibilita()), righe=len(base_rows),
            W_per_seme={str(SEME): stima_W(base_rows, MISURE["p_UNKNOWN (principale)"], 0)["stima"],
                        **{str(s): stima_W(v, MISURE["p_UNKNOWN (principale)"], 0)["stima"] for s, v in ripet.items()}},
            risposte_uguali_al_seme_principale={str(s): sum(o["parsed"] == risposte0[o["row_id"]] for o in v)
                                                for s, v in ripet.items()})
    sintesi["esiti"] = dict(esiti, **fermati)
    ragion = [dict(row_id=o["row_id"], condizione=o["condizione"], seme=o.get("seme", ""), chiusura=o["chiusura"],
                   ragionamento_token=o["ragionamento_token"], ragionamento=o.get("ragionamento", ""),
                   risposta=o["generated"]) for o in righe_out if o["condizione"] == "acceso"]
    rip_out = [dict(o, ragionamento=None) for v in ripet.values() for o in v]
    punteggi = [{k: v for k, v in o.items() if k != "ragionamento"} for o in righe_out]
    anomalie = [dict(a, condizione=c) for c, e in esiti.items() for a in e["livelli"]["anomalie"]]
    archivio = esporta(cartella, sintesi, dict(punteggi=punteggi, ragionamenti=ragion, ripetibilita=rip_out,
                                                scarti_lunghezza=[dict(s, condizione=c) for c, v in scarti.items() for s in v],
                                                anomalie=anomalie))
    print(f"  {nome}: " + " | ".join(f"{c} {e['esito']} (W {e['report_indetbench']['W_p_unknown']['value']})"
                                     for c, e in esiti.items()) + f" | salvato {archivio.name}")
    return cartella
print("Funzioni GPU definite (nessun modello caricato).")

In [ ]:
# 6. Avvio esplicito. Se la sessione cade, rilancia tutte le celle: i blocchi già salvati su Drive non si ricalcolano.
def riga(nome, cond, e, s):
    rep = e.get("report_indetbench")
    if not rep:
        return dict(modello=nome, condizione=cond, esito=e["esito"])
    g = rep["competence_gate"]; w = rep["W_p_unknown"]; base = e["baseline_dalle_uscite"]
    rg = e.get("ragionamento", {})
    return dict(modello=nome, condizione=cond, esito=e["esito"], W=w["value"], IC95_basso=w["ci_lower"],
                IC95_alto=w["ci_upper"], coppie=w["n_pairs"], acc_B=g["accuracy_B"],
                acc_B_gold_NO=g["accuracy_B_by_gold"]["NO"], acc_B_gold_YES=g["accuracy_B_by_gold"]["YES"],
                soglia_superata=g["passed"], massa_mediana=e.get("massa_mediana_tutte"),
                W_entropia=base["entropia normalizzata"]["W"]["stima"],
                W_meno_p_max=base["meno p massima normalizzata"]["W"]["stima"],
                W_risposta_UNKNOWN=base["risposta generata UNKNOWN (0/1)"]["W"]["stima"],
                W_lunghezza_ragionamento=base["ragionamento: lunghezza in token"]["W"]["stima"],
                UNK_B=e["quota_UNKNOWN"]["B"], UNK_C=e["quota_UNKNOWN"]["C"],
                coppie_chiuse_da_sole=rg.get("coppie_chiuse_da_sole", ""),
                token_mediana_B=(rg.get("token_mediana") or {}).get("B", ""),
                token_mediana_C=(rg.get("token_mediana") or {}).get("C", ""),
                HR2_delta_W=(s.get("HR2_delta_W_acceso_meno_spento") or {}).get("stima", "") if cond == "acceso" else "")

if AVVIA_GPU:
    if not SMOKE and not CONTROLLO_CPU["superato"]:
        raise RuntimeError("Completo fermato: il controllo CPU dei 4 W del Test U2 (cella 4b) non è superato.")
    nome_gpu, fam = famiglia_gpu()
    print("GPU:", nome_gpu, f"| memoria {memoria_gpu_gb():.0f} GB")
    if SOLO_A100 and fam != "A100":
        raise RuntimeError(f"GPU {nome_gpu}: serve una A100 (SOLO_A100 = True)")
    if not SMOKE and MODELLI and MODELLI[0] != "qwen3_8b":
        raise RuntimeError("Nel completo Qwen3-8B va per primo: il suo controllo della pipeline (§5.1) decide se si parte.")
    token = hf_token()
    cartelle, errori = {}, {}
    for nome in MODELLI:
        print(f"\n=== {nome}: {MODEL_CONFIG[nome]['id']} @ {MODEL_CONFIG[nome]['sha'][:7]} | condizioni "
              f"{', '.join(MODEL_CONFIG[nome]['condizioni'])} ===")
        try:
            cartelle[nome] = valuta_modello(nome, token)
        except Exception as err:
            errori[nome] = nascondi(f"{type(err).__name__}: {err}")[:500]
            print(f"  ERRORE {nome}: {errori[nome]}")
            if nome == "qwen3_8b" and not SMOKE:
                print("  Il completo si ferma qui: Qwen3-8B non ha concluso, il controllo della pipeline (§5.1) manca.")
                break
            continue
        if nome == "qwen3_8b" and not SMOKE:
            cp = json.loads((cartelle[nome] / "esito.json").read_text()).get("controllo_pipeline") or {}
            if not cp.get("superato"):
                errori[nome] = "controllo della pipeline (§5.1) mancante o non superato"
                print(f"  {errori[nome]}: il completo si ferma qui.")
                break
    sintesi = {n: json.loads((c / "esito.json").read_text()) for n, c in cartelle.items()}
    righe = [riga(n, c, e, s) for n, s in sintesi.items() for c, e in s.get("esiti", {"—": s}).items()]
    righe += [dict(modello=k, esito=f"ERRORE: {v}") for k, v in errori.items()]
    note = ["I risultati della parte B sono ESPLORATIVI: non sono una validazione indipendente del metodo.",
            "p(UNKNOWN) con ragionamento è misurata al primo token della risposta dopo </think> ed è condizionata al "
            "ragionamento campionato: leggerla insieme alla ripetibilità.",
            "HR2 (acceso − spento) vale solo a pesi fissi (Qwen3); Phi-4-reasoning e R1-Distill sono descrittivi.",
            "Un W più alto con il ragionamento non dimostra che il metodo funzioni meglio, e uno più basso non lo smentisce.",
            NOTA_NORMALIZZATA]
    if SMOKE:
        note.insert(0, "MODALITÀ PROVA: verifica tecnica e misura dei tempi, nessuna lettura scientifica.")
    write_json(ROOT / "riepilogo.json", dict(versione=VERSIONE, modalita=MODALITA, repo_commit=REPO_COMMIT, note=note,
                                              righe=righe, errori=errori, controllo_cpu_testU2=CONTROLLO_CPU))
    scrivi_csv(ROOT / "riepilogo.csv", righe)
    parti = [f"IndetBench, parte B (ragionamento) | {VERSIONE} | modalità {MODALITA} | repository {REPO_COMMIT[:7]}", ""] + note
    for n, s in sintesi.items():
        parti += ["", f"===== {n} ({s.get('modello')}) | {s.get('ruolo', '')} ====="]
        for k in ("controllo_pipeline", "spento_contro_test_U2", "copertura_etichette_24_righe",
                  "HR2_delta_W_acceso_meno_spento", "risposte_spento_acceso", "ripetibilita", "secondi_per_condizione"):
            if k in s:
                parti.append(f"{k} {json.dumps(s[k], ensure_ascii=False)}")
        for c, e in s.get("esiti", {}).items():
            parti.append(f"--- condizione {c} | esito: {e['esito']}")
            if "ragionamento" in e:
                parti.append(f"ragionamento {json.dumps(e['ragionamento'], ensure_ascii=False)}")
            for livello, contenuto in e.get("livelli", {}).items():
                if livello == "anomalie":
                    parti.append(f"Anomalie da classificare ({len(contenuto)}):")
                    parti += [f"  - {a['anomalia']} [{a['prova']}] possibili: {a['categorie_possibili']}" for a in contenuto]
                else:
                    parti.append(f"{livello} {json.dumps(contenuto, ensure_ascii=False)}")
        if "esiti" not in s:
            parti.append(f"esito: {s.get('esito')}")
    if SMOKE and "qwen3_8b" in sintesi and "secondi_per_condizione" in sintesi["qwen3_8b"]:
        sec = sintesi["qwen3_8b"]["secondi_per_condizione"]
        fattore = 1600 / max(len(RIGHE), 1)
        parti += ["", "Stima dei tempi del completo dalla prova (Qwen3-8B, stesse impostazioni): "
                  + ", ".join(f"{c} ≈ {v * fattore / 3600:.1f} h" for c, v in sec.items())
                  + ". Phi-4-reasoning e Qwen3-32B sono più lenti: la stima va rifatta con i numeri della prova."]
    if errori:
        parti += ["", "Errori:"] + [f"  {k}: {v}" for k, v in errori.items()]
    testo = "\n".join(parti)
    (ROOT / "rapporto.txt").write_text(testo, encoding="utf-8")
    finale = ROOT / f"indetbench_reasoning_{MODALITA}.zip"
    with zipfile.ZipFile(finale.with_suffix(".tmp"), "w", zipfile.ZIP_DEFLATED) as z:
        for n in ("riepilogo.json", "riepilogo.csv", "rapporto.txt", "controllo_cpu_testU2.json"):
            z.write(ROOT / n, n)
        for n, c in cartelle.items():
            for f in sorted(c.glob("*.json")) + sorted(c.glob("*.csv")):
                z.write(f, f"{c.name}/{f.name}")
    os.replace(finale.with_suffix(".tmp"), finale)
    print("\n" + testo)
    print(f"\nZIP: {finale}")
else:
    print("Preflight concluso. Imposta AVVIA_GPU = True per eseguire la modalità selezionata.")